In [1]:
import torch
from monai.networks.nets import SwinUNETR
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
model = SwinUNETR(
    img_size=(96,96,96),
    in_channels=3,
    out_channels=3
).to(device)

In [3]:
import os

image_dir = "/Users/apple/Desktop/BRAINIAC/data/processed/images"
mask_dir = "/Users/apple/Desktop/BRAINIAC/data/processed/masks"

images = sorted([
    os.path.join(image_dir, f)
    for f in os.listdir(image_dir)
    if f.endswith(".npy")
])

masks = sorted([
    os.path.join(mask_dir, f)
    for f in os.listdir(mask_dir)
    if f.endswith(".npy")
])

data = [
    {"image": img, "label": mask}
    for img, mask in zip(images, masks)
]

print("Total samples:", len(data))

Total samples: 368


In [4]:
from monai.transforms import Compose, LoadImaged, Lambdad, Resized, ToTensord

transforms = Compose([
    
    LoadImaged(keys=["image","label"]),

    Lambdad(keys=["image"], func=lambda x: x.transpose(3,0,1,2)),

    Lambdad(keys=["label"], func=lambda x: x[np.newaxis,...]),

    Resized(keys=["image","label"], spatial_size=(96,96,96)),

    ToTensord(keys=["image","label"])

])

In [5]:
def compute_entropy_attention(att_maps):

    if len(att_maps) == 0:
        return torch.zeros(1,1,96,96,96).to(device)

    att = torch.mean(torch.stack(att_maps).detach(), dim=0)

    att = torch.softmax(att, dim=-1)

    entropy = -torch.sum(att * torch.log(att + 1e-8), dim=-1)

    entropy = entropy.unsqueeze(1)

    entropy = (entropy - entropy.min()) / (entropy.max() - entropy.min() + 1e-8)

    return entropy.to(device)

In [6]:
def compute_css_map(model, x, epsilon=0.01):

    noise = epsilon * torch.randn_like(x)
    x_cf = x + noise

    with torch.no_grad():
        y1 = model(x)
        y2 = model(x_cf)

    css = torch.abs(y2 - y1)
    css = css.mean(dim=1, keepdim=True)

    css = torch.nn.functional.interpolate(
        css,
        size=x.shape[2:],
        mode="trilinear",
        align_corners=False
    )

    css = (css - css.min()) / (css.max() - css.min() + 1e-8)

    return css

In [7]:
# Counterfactual Sensitivity Score (CSS)

def compute_css_map(model, x, epsilon=0.01):

    noise = epsilon * torch.randn_like(x)

    x_cf = x + noise

    with torch.no_grad():
        y1 = model(x)
        y2 = model(x_cf)

    css = torch.abs(y2 - y1)

    css = css.mean(dim=1, keepdim=True)

    css = torch.nn.functional.interpolate(
        css,
        size=x.shape[2:],
        mode="trilinear",
        align_corners=False
    )

    css = (css - css.min()) / (css.max() - css.min() + 1e-8)

    return css

In [8]:
# Feature Fusion (Encoder + Entropy + CSS)

import torch.nn.functional as F

def fuse_features(enc, entropy, css):

    entropy = F.interpolate(entropy, size=enc.shape[2:], mode="trilinear", align_corners=False)
    css = F.interpolate(css, size=enc.shape[2:], mode="trilinear", align_corners=False)

    fused = torch.cat([enc, entropy, css], dim=1)

    return fused

In [9]:
class GradCAM:
    def __init__(self, model, target_layer):
        self.model = model
        self.target_layer = target_layer

        self.activations = None
        self.gradients = None

        # forward hook
        def forward_hook(module, input, output):
            self.activations = output

        # backward hook
        def backward_hook(module, grad_input, grad_output):
            self.gradients = grad_output[0]

        self.target_layer.register_forward_hook(forward_hook)
        self.target_layer.register_backward_hook(backward_hook)

    def generate(self):

        if self.activations is None or self.gradients is None:
            raise RuntimeError("Hooks did not capture activations/gradients")

        weights = torch.mean(self.gradients, dim=(2,3,4), keepdim=True)

        cam = torch.sum(weights * self.activations, dim=1)

        cam = torch.relu(cam)

        cam = torch.nn.functional.interpolate(
            cam.unsqueeze(1),
            size=(96,96,96),
            mode="trilinear",
            align_corners=False
        )

        cam = cam.squeeze()

        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)

        return cam.detach().cpu()

In [10]:
# Visual overlay plot

import matplotlib.pyplot as plt

def visualize_explainability(x, entropy, css, cam):

    slice_idx = x.shape[-1] // 2

    x_slice = x[0,0,:,:,slice_idx].cpu()

    ent_slice = entropy[0,0,:,:,slice_idx].cpu()

    css_slice = css[0,0,:,:,slice_idx].cpu()

    cam_slice = cam[0,:,:,slice_idx].cpu()

    fig, axs = plt.subplots(1,4, figsize=(18,5))

    axs[0].imshow(x_slice, cmap="gray")
    axs[0].set_title("Original MRI")

    axs[1].imshow(ent_slice, cmap="hot")
    axs[1].set_title("Entropy Attention")

    axs[2].imshow(css_slice, cmap="hot")
    axs[2].set_title("CSS Counterfactual")

    axs[3].imshow(cam_slice, cmap="hot")
    axs[3].set_title("GradCAM")

    for ax in axs:
        ax.axis("off")

    plt.show()

In [11]:
import os

image_dir = "/Users/apple/Desktop/BRAINIAC/data/processed/images"
mask_dir = "/Users/apple/Desktop/BRAINIAC/data/processed/masks"

images = sorted([
    os.path.join(image_dir, f)
    for f in os.listdir(image_dir)
    if f.endswith(".npy")
])

masks = sorted([
    os.path.join(mask_dir, f)
    for f in os.listdir(mask_dir)
    if f.endswith(".npy")
])

data = [
    {"image": img, "label": mask}
    for img, mask in zip(images, masks)
]

print("Total samples:", len(data))

Total samples: 368


In [12]:
import torch
import gc

# limit CPU usage (prevents kernel crash on Mac)
torch.set_num_threads(2)

# free unused memory
gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

In [13]:
from torch.utils.data import DataLoader
import torch
from torch.utils.data import DataLoader

In [14]:
from monai.data import Dataset as MonaiDataset

split = int(len(data) * 0.8)

train_files = data[:split]
val_files = data[split:]

train_dataset = MonaiDataset(data=train_files, transform=transforms)
val_dataset = MonaiDataset(data=val_files, transform=transforms)

In [15]:
train_loader = DataLoader(
    train_dataset,
    batch_size=1,      # keep very small
    shuffle=True,
    num_workers=0,     # important for Mac stability
    pin_memory=False
)

In [16]:
attention_maps = []
feature_maps = []

In [17]:
cam_engine = GradCAM(
    model,
    target_layer=list(model.modules())[-5]
)

In [19]:
# Run Explainability

model.eval()

attention_maps = []
feature_maps = []

x = torch.randn(1,3,96,96,96).to(device)

model.zero_grad()

# Forward pass
y_hat = model(x)

# Compute entropy attention
entropy = compute_entropy_attention(attention_maps)

# Compute counterfactual sensitivity
css = compute_css_map(model, x)

# GradCAM
target = torch.argmax(y_hat)

loss = y_hat.flatten()[target]

loss.backward()

cam = cam_engine.generate()

# Visualize results
visualize_explainability(x, entropy, css, cam)

IndexError: too many indices for tensor of dimension 3